## Processamento Digital de Imagens - PDI - 2026.1

### Lab 7 - Processamento Morfológico de Imagens

*Morphological Image Processing*
* Gonzalez, capitulo 9

- - - - - 

**Preparativos iniciais**:
Importar bibliotecas necessárias e criar uma função auxiliar para visualização de imagens.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import requests

def exibir_imagens(imagens, titulos):
    """
    Exibe uma lista de imagens lado a lado usando matplotlib.
    """
    n = len(imagens)
    fig, axes = plt.subplots(1, n, figsize=(15, 5))
    if n == 1:
        axes = [axes]

    for ax, img, tit in zip(axes, imagens, titulos):
        ax.imshow(img, cmap='gray')
        ax.set_title(tit)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

print("Função útil para exibir imagens.")

**Carregamento e Criação de Imagens para o exercicio**

- Gerar a imagem clássica do 'J' via matrizes Numpy
- Ler imagens de moedas
- Baixar formas geométricas reais via URL.


In [ ]:
import cv2
import numpy as np
import requests

def download_image(url):
    resp = requests.get(url)
    if resp.status_code == 200:
        img_array = np.frombuffer(resp.content, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
        return img
    else:
        print(f"Falha ao baixar: {url}")
        return None

# URL fornecida
url_smarties = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/smarties.png"

# 1. Ler imagem. Baixar e decodificar a imagem
img_coins = cv2.cvtColor(cv2.imread('water_coins.png'), cv2.COLOR_BGR2GRAY)
img_smarties = download_image(url_smarties)

# 2. Criar uma imagem binária simples do 'J' via matrizes Numpy
img_j = np.zeros((300, 300), dtype="uint8")
cv2.putText(img_j, 'J', (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 10, 255, 25)

# 3. Exibir os resultados
imagens_para_exibir = [img for img in [img_coins, img_smarties, img_j] if img is not None]
titulos_para_exibir = ["Moedas","Smarties", "J Manual"]

exibir_imagens(imagens_para_exibir, titulos_para_exibir)

### Parte 1: Binarização e Limiarização

Aplicar técnicas de thresholding (limiarização) global e o método de Otsu para converter as imagens em preto e branco (binárias).


In [ ]:
# 1. Binarização global simples em 'img_smarties'
threshold_value = 127
_, img_smarties_threshold = cv2.threshold(img_smarties, threshold_value, 255, cv2.THRESH_BINARY)

# 2. Método de Otsu em 'img_smarties'
# O método de Otsu calcula o limiar ideal automaticamente
_, img_coins_otsu = cv2.threshold(img_coins, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
_, img_smarties_otsu = cv2.threshold(img_smarties, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 3. Garantir que 'img_j' seja binária e consistente
_, img_j_bin = cv2.threshold(img_j, 1, 255, cv2.THRESH_BINARY)

# 4. Exibir os resultados das binarizações
bin_images = [img_smarties, img_smarties_threshold, img_smarties_otsu, img_coins_otsu, img_j_bin]
bin_titles = ["Smarties Original", "Global Threshold (127)", "Otsu Thresholding", "Moedas bin", "J bin"]

exibir_imagens(bin_images, bin_titles)

print("Binarização Método Manual e Método OTSU adaptativo.")

- - - - -



*   **Insight:** O método de Otsu demonstrou ser superior ao limiar global fixo para imagens com iluminação variável, como a dos "Smarties".

- - - - -
### Parte 2: **Elementos Estruturantes (Kernels)**: 
Criar diferentes tipos de elementos estruturantes (retangular, elíptico e em cruz) usando NumPy e OpenCV.

In [ ]:
import cv2
import numpy as np

# 1 & 2. Criar kernels usando cv2.getStructuringElement (tamanho 5x5)
kernel_rect = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
kernel_ellip = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
kernel_cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))

# 3. Criar kernel retangular simples via NumPy para comparação
kernel_numpy = np.ones((5, 5), np.uint8)

# 4. Imprimir as matrizes para visualização
print("Kernel Retangular (OpenCV):\n", kernel_rect)
print("\nKernel Elíptico:\n", kernel_ellip)
print("\nKernel em Cruz:\n", kernel_cross)
print("\nKernel NumPy (ones):\n", kernel_numpy)

print("\nKernels definidos.")

- - - - -
### **Parte 3: Operações de Erosão e Dilatação**: 
Aplicar as funções `cv2.erode` e `cv2.dilate` nas imagens binarizadas `img_j_bin` - `img_smarties_otsu` - `img_coins_otsu` para analisar os efeitos de afinamento e espessamento.

In [ ]:
import cv2
import numpy as np

# 1. Erosão na imagem 'img_j_bin'
img_j_erosion = cv2.erode(img_j_bin, kernel_rect, iterations=1)

# 2. Dilatação na imagem 'img_j_bin'
img_j_dilation = cv2.dilate(img_j_bin, kernel_rect, iterations=1)

# 3. Erosão e Dilatação na imagem 'img_smarties_otsu' usando kernel ELIPTICO
img_smarties_erosion = cv2.erode(img_smarties_otsu, kernel_ellip, iterations=1)
img_smarties_dilation = cv2.dilate(img_smarties_otsu, kernel_ellip, iterations=1)


# 4. Erosão e Dilatação na imagem 'img_coins_otsu' usando kernel CRUZ
img_coins_erosion = cv2.erode(img_coins_otsu, kernel_cross, iterations=1)
img_coins_dilation = cv2.dilate(img_coins_otsu, kernel_cross, iterations=1)

# 5. Visualização comparativa para o conjunto 'J'
exibir_imagens(
    [img_j_bin, img_j_erosion, img_j_dilation],
    ['J Original Binário', 'J Erosão (Afinamento)', 'J Dilatação (Espessamento)']
)

# Visualização comparativa para o conjunto 'Smarties'
exibir_imagens(
    [img_smarties_otsu, img_smarties_erosion, img_smarties_dilation],
    ['Smarties Otsu', 'Smarties Erosão', 'Smarties Dilatação']
)

# Visualização comparativa para o conjunto 'MOEDAS'
exibir_imagens(
    [img_coins_otsu, img_coins_erosion, img_coins_dilation],
    ['Moedas Otsu', 'Moedas Erosão', 'Moedas Dilatação']
)

# 5. Confirmação
print("Operações de erosão e dilatação: efeitos de afinamento e espessamento.")

- - - - - 
**Extração de Fronteiras (Gradiente Morfológico)**:
Implementar a extração de fronteiras subtraindo a imagem erosionada da imagem original, e também demonstrar o uso da função `cv2.morphologyEx` com o parâmetro `MORPH_GRADIENT`.

In [ ]:
# 1. Extração manual da fronteira para 'img_j_bin'
# Subtraindo a imagem erosionada da original
img_j_boundary_manual = cv2.subtract(img_j_bin, img_j_erosion)
img_coins_boundary_manual = cv2.subtract(img_coins_otsu, img_coins_erosion)

# 2. Gradiente morfológico via cv2.morphologyEx para 'img_j_bin'
img_j_gradient = cv2.morphologyEx(img_j_bin, cv2.MORPH_GRADIENT, kernel_rect)

# 3. Gradiente morfológico via cv2.morphologyEx para 'img_smarties_otsu'
# Usando o kernel elíptico para melhor conformidade com as formas circulares
img_smarties_gradient = cv2.morphologyEx(img_smarties_otsu, cv2.MORPH_GRADIENT, kernel_ellip)
img_coins_gradient = cv2.morphologyEx(img_coins_otsu, cv2.MORPH_GRADIENT, kernel_cross)

# 4. Visualização dos resultados
# Comparação para a imagem 'J'
exibir_imagens(
    [img_j_bin, img_j_boundary_manual, img_j_gradient],
    ['J Original', 'Fronteira Manual (Subtração)', 'Fronteira (MORPH_GRADIENT)']
)

# Comparação para a imagem 'Smarties'
exibir_imagens(
    [img_smarties_otsu, img_smarties_gradient],
    ['Smarties Otsu Original', 'Smarties Gradiente Morfológico']
)

# Comparação para a imagem 'J'
exibir_imagens(
    [img_coins_otsu, img_coins_boundary_manual, img_coins_gradient],
    ['Moedas Original', 'Fronteira Manual (Subtração)', 'Moedas Gradiente Morfológico']
)

print("Extração de fronteiras utilizando métodos manuais e automáticos.")

- - - - -
**Análise Comparativa de Bordas**:
Visualizar e comparar as fronteiras obtidas com diferentes tamanhos de kernel para entender o controle da espessura da borda extraída.

In [ ]:
sizes = [3, 7, 11]

# 1. Criar kernels retangulares e elípticos de diferentes tamanhos
kernels_j = [cv2.getStructuringElement(cv2.MORPH_RECT, (s, s)) for s in sizes]
kernels_s = [cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (s, s)) for s in sizes]
kernels_c = [cv2.getStructuringElement(cv2.MORPH_CROSS, (s, s)) for s in sizes]

# 2. Aplicar gradiente morfológico na imagem 'J'
gradients_j = [cv2.morphologyEx(img_j_bin, cv2.MORPH_GRADIENT, k) for k in kernels_j]

# 3. Aplicar gradiente morfológico na imagem 'Smarties'
gradients_s = [cv2.morphologyEx(img_smarties_otsu, cv2.MORPH_GRADIENT, k) for k in kernels_s]

# 4. Aplicar gradiente morfológico na imagem 'Moedas'
gradients_c = [cv2.morphologyEx(img_coins_otsu, cv2.MORPH_GRADIENT, k) for k in kernels_c]

# 5. Visualização comparativa
titles = [f'Kernel {s}x{s}' for s in sizes]

print("Comparação de Bordas - Imagem J:")
exibir_imagens(gradients_j, [f'J - {t}' for t in titles])

print("\nComparação de Bordas - Imagem Smarties:")
exibir_imagens(gradients_s, [f'Smarties - {t}' for t in titles])

print("\nComparação de Bordas - Imagem Moedas:")
exibir_imagens(gradients_c, [f'Moedas - {t}' for t in titles])

print("Análise visual: o aumento do tamanho do kernel resulta em bordas mais espessas.")

- - - - -
###  **Parte 4: Operações de Abertura e Fechamento Morfológico:**

A **Abertura (Opening)** é uma erosão seguida por uma dilatação, útil para remover ruídos pequenos e desconectar objetos ligeiramente conectados, preservando o tamanho e a forma de objetos maiores. 

O **Fechamento (Closing)** é uma dilatação seguida por uma erosão, útil para preencher pequenos buracos e quebras em objetos, e suavizar as suas fronteiras, preservando o tamanho e a forma de objetos maiores.

- - - - -
Utilização da função `cv2.morphologyEx()` para Abertura e Fechamento.

In [ ]:
import cv2
import numpy as np

# 1. Operação de Abertura (Opening)
# Remove pequenos objetos brancos (ruído) e desconecta objetos

# Abertura em img_j_bin (kernel retangular)
img_j_opened = cv2.morphologyEx(img_j_bin, cv2.MORPH_OPEN, kernel_rect)

# Abertura em img_smarties_otsu (kernel elíptico)
img_smarties_opened = cv2.morphologyEx(img_smarties_otsu, cv2.MORPH_OPEN, kernel_ellip)

# Abertura em img_coins_otsu (kernel cruz)
img_coins_opened = cv2.morphologyEx(img_coins_otsu, cv2.MORPH_OPEN, kernel_cross)

# 2. Operação de Fechamento (Closing)
# Preenche pequenos buracos dentro dos objetos e conecta componentes próximos

# Fechamento em img_j_bin (kernel retangular)
img_j_closed = cv2.morphologyEx(img_j_bin, cv2.MORPH_CLOSE, kernel_rect)

# Fechamento em img_smarties_otsu (kernel elíptico)
img_smarties_closed = cv2.morphologyEx(img_smarties_otsu, cv2.MORPH_CLOSE, kernel_ellip)

# Fechamento em img_coins_otsu (kernel cruz)
img_coins_closed = cv2.morphologyEx(img_coins_otsu, cv2.MORPH_CLOSE, kernel_cross)

# 3. Visualização dos resultados

exibir_imagens(
    [img_j_bin, img_j_opened, img_j_closed],
    ['J Original Binário', 'J Abertura', 'J Fechamento']
)

exibir_imagens(
    [img_smarties_otsu, img_smarties_opened, img_smarties_closed],
    ['Smarties Otsu Original', 'Smarties Abertura', 'Smarties Fechamento']
)

exibir_imagens(
    [img_coins_otsu, img_coins_opened, img_coins_closed],
    ['Moedas Otsu Original', 'Moedas Abertura', 'Moedas Fechamento']
)

print("Operações de abertura e fechamento morfológico.")

- - - - -
Aplicação prática usual de **Opening**

In [ ]:
import cv2
import numpy as np

# 1. Ler a imagem clássica da Impressão Digital (Fingerprint)
img_fingerprint = cv2.imread('noisy_fingerprint.png')

# 2. Aplicar Abertura para Limpeza de Ruído
# Abertura = Erosão (elimina ruído pequeno) -> Dilatação (restaura o objeto principal)
kernel_fp = np.ones((3, 3), np.uint8)
img_fp_opened = cv2.morphologyEx(img_fingerprint, cv2.MORPH_OPEN, kernel_fp)

# 3. Visualização
exibir_imagens(
    [img_fingerprint, img_fp_opened],
    ['Digital Original (Ruidosa)', 'Após Abertura (Vantagem: Limpeza)']
)

print("Exemplo de Abertura para Remoção de ruído preservando a estrutura principal.")

- - - - -
Aplicação prática usual de **Closing**

In [ ]:
import cv2
import numpy as np

# 1.1 Criar uma imagem simulando caracteres fragmentados (exemplo clássico acadêmico)
img_text = np.zeros((200, 600), dtype='uint8')
cv2.putText(img_text, 'TEXTO', (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 5, 255, 20)

# Simular falhas: adicionar ruído preto (buracos) sobre o texto branco
noise = np.random.randint(0, 2, (200, 600), dtype='uint8') * 255
img_text_broken = cv2.bitwise_and(img_text, cv2.bitwise_not(noise))

# 1.2 Ler a imagem de texto com falhas
img_txtgaps = cv2.cvtColor(cv2.imread('text_gaps_.png'), cv2.COLOR_BGR2GRAY)

# 2. Aplicar Fechamento (Closing)
# Fechamento = Dilatação (conecta partes próximas) -> Erosão (restaura o tamanho)
kernel_close = np.ones((5, 5), np.uint8)
img_text_closed = cv2.morphologyEx(img_text_broken, cv2.MORPH_CLOSE, kernel_close)
kernel_close2 = np.ones((3, 3), np.uint8)
img_txtgaps_closed = cv2.morphologyEx(img_txtgaps, cv2.MORPH_CLOSE, kernel_close2)

# 3. Visualização
exibir_imagens(
    [img_text_broken, img_text_closed],
    ['Texto com Falhas (Original)', 'Após Fechamento (Vantagem: Conexão)']
)

exibir_imagens(
    [img_txtgaps, img_txtgaps_closed],
    ['Texto 2 com Falhas (Original)', 'Texto 2 após Fechamento']
)

print("Fechamento: Preenchimento de falhas e conexão de componentes fragmentados.")

- - - - -